In [9]:
feature_name = 'ecfp_feature'
important_features = [30, 123, 10, 16, 81, 33]
important_features_simple = [f'{feature_name}_{i}' for i in important_features]
important_features_piecewise_interactions = [f'{feature_name}_{i}' for i in important_features] + [f'{feature_name}_10 x {feature_name}_{j}' for j in important_features if j != 10]
important_feature_nonlinear_interactions = [f'{feature_name}_{i}' for i in important_features] + [f'{feature_name}_30 x {feature_name}_123']

choices = {
    'qm9_simple_linear6': important_features_simple,
    'qm9_piecewise_linear_6': important_features_piecewise_interactions,
    'qm9_nonlinear_6': important_feature_nonlinear_interactions
}

In [10]:
from sklearn.preprocessing import StandardScaler
import pandas as pd

class GtModel:
    def __init__(self, dataset_name, feature_order, X_train):
        self.dataset_name = dataset_name
        self.feature_order = feature_order
        self.selected_features_names = [30, 123, 10, 16, 81, 33]
        self.selected_features_positions = [self.feature_order[f'ecfp_feature_{i}'] for i in self.selected_features_names]
        self.choices = {
            'qm9_simple_linear6': self.linear_function,
            'qm9_piecewise_linear_6': self.piecewise_linear_function,
            'qm9_nonlinear_6': self.nonlinear_function,
        }
        self.scaler = StandardScaler().fit(X_train.to_numpy())
        self.X_train = X_train

    def linear_function(self, df, df_not_scaled):
        f_30 = df[:, self.selected_features_positions[0]]
        f_123 = df[:, self.selected_features_positions[1]]
        f_10 = df[:, self.selected_features_positions[2]]
        f_16 = df[:, self.selected_features_positions[3]]
        f_81 = df[:, self.selected_features_positions[4]]
        f_33 = df[:, self.selected_features_positions[5]]
        target = 8.5 * f_30 + 10.5 * f_123 - 3.5 * f_10 + 3 * f_16 - 2.5 * f_81 + 5.5 * f_33 + 30
        return target

    def piecewise_linear_function(self, df, df_not_scaled):
        f_30 = df[:, self.selected_features_positions[0]]
        f_123 = df[:, self.selected_features_positions[1]]
        f_16 = df[:, self.selected_features_positions[3]]
        f_81 = df[:, self.selected_features_positions[4]]
        f_33 = df[:, self.selected_features_positions[5]]

        f_10 = df_not_scaled[:, self.selected_features_positions[2]]
        f_10 = f_10.round(0)

        conditions = [
            f_10 < 1,
            (f_10 >= 1) & (f_10 < 2),
            f_10 >= 2
        ]

        choices = [
            10.5 * f_30 + 6.5 * f_123 - 1.5 * f_81 + 30,
            5 * f_30 + 13 * f_123 - 2.5 * f_16 + 30,
            -1.5 * f_30 + 3.5 * f_123 + 15.5 * f_33 + 30
        ]
        target = np.select(conditions, choices, default=10000)
        return target

    def nonlinear_function(self, df, df_not_scaled):
        f_30 = df[:, self.selected_features_positions[0]]
        f_123 = df[:, self.selected_features_positions[1]]
        f_10 = df[:, self.selected_features_positions[2]]
        f_16 = df[:, self.selected_features_positions[3]]
        f_81 = df[:, self.selected_features_positions[4]]
        f_33 = df[:, self.selected_features_positions[5]]

        component1 = -9.5 * f_10 + 2.5 * f_81 ** 2 - 3.5 * f_16
        component2 = 7.5 * f_30 * f_123
        component3 = 1.5 * f_33 ** 2 + 30
        return component1 + component2 + component3

    def predict(self, df):
        df_scaled = self.scaler.transform(df)
        if isinstance(df, pd.DataFrame) or isinstance(df, pd.Series):
            df = df.to_numpy()
            df_scaled = df_scaled.to_numpy()
        return self.choices[self.dataset_name](df_scaled, df)

In [11]:
import numpy as np
from src.analysis.processing import lime_ranking, shap_ranking, shapiq_ranking, meg_ranking, mmace_ranking, \
    meg_cf_percent, mmace_cf_percent
from src.analysis.xai_eval import pgi, pgu, feature_agreement
import pickle
import os
import joblib

dataset_name = 'qm9_piecewise_linear_6'
results_dir = f'../results/gt_synthetic_data/{dataset_name}/explanations'
model_dir = f'../results/gt_synthetic_data/{dataset_name}/'

target = 'target'

results_dict = {
    'lime': ('lime_results.pickle', lime_ranking),
    'shap': ('shap_results.pickle', shap_ranking),
    'shapiq1': ('shapiq1_results.pickle', shapiq_ranking),
    'shapiq2': ('shapiq2_results.pickle', shapiq_ranking),
    'meg': ('meg_results.pickle', meg_ranking, meg_cf_percent),
    'mmace': ('mmace_results.pickle', mmace_ranking, mmace_cf_percent),
}

ranking_dict = {}
ranking_per_fold_dict = {}
cf_similarity_dict = {}
cf_validity_dict = {}
metrics_dict = {}
metrics_top10_dict = {}

for key in results_dict.keys():
    print(key)
    file_name, ranking_func = results_dict[key][:2]

    if len(results_dict[key]) > 2:
        cf_func = results_dict[key][2]
    else:
        cf_func = None
    with open(os.path.join(results_dir, file_name), 'rb') as f:
        results = pickle.load(f)

    ranking, rankings_per_fold = ranking_func(results, target)
    display(ranking.head(20))
    if cf_func is not None:
        cf_percent = cf_func(results, target=target)
    else:
        cf_percent = None

    if cf_percent is not None:
        print(f"{key} counterfactual percent: {cf_percent}")

    pgis, pgus = [], []
    pgis_10, pgus_10 = [], []
    pgis_org, pgus_org = [], []
    pgis_org_10, pgus_org_10 = [], []
    fas = []
    fas_interactions = []

    ranking_dict[key] = ranking
    ranking_per_fold_dict[key] = rankings_per_fold
    if cf_percent is not None:
        cf_validity_dict[key] = cf_percent[0]
        cf_similarity_dict[key] = cf_percent[1]

    for i in range(len(rankings_per_fold)):
        model = os.path.join(model_dir, f'model_{i}.joblib')
        model = joblib.load(model)
        test_examples = results['test_data'][i].drop(columns=[target])
        train_examples = results['training_data'][i].drop(columns=[target])

        ranking_current = list(rankings_per_fold[i]['features'])
        print(ranking_current[:6])

        pgi_one, pgi_org = pgi(test_examples, ranking_current, model, train_examples)
        pgu_one, pgu_org = pgu(test_examples, ranking_current, model, train_examples)
        pgi_ten, pgi_ten_org = pgi(test_examples, ranking_current, model, train_examples, len_max=6)
        pgu_ten, pgu_ten_org = pgu(test_examples, ranking_current, model, train_examples, len_max=6)
        pgis.append(pgi_one)
        pgus.append(pgu_one)
        pgis_10.append(pgi_ten)
        pgus_10.append(pgu_ten)
        pgis_org.append(pgi_org)
        pgus_org.append(pgu_org)
        pgis_org_10.append(pgi_ten_org)
        pgus_org_10.append(pgu_ten_org)

        fa = feature_agreement(choices['qm9_simple_linear6'], ranking_current)
        fas.append(fa)
        if key == 'shapiq2':
            fa_interactions = feature_agreement(choices[dataset_name], ranking_current)
            fas_interactions.append(fa_interactions)

    pgi_mean = np.mean(pgis)
    pgu_mean = np.mean(pgus)
    pgi_std = np.std(pgis)
    pgu_std = np.std(pgus)
    pgi_org_mean = np.mean(pgis_org)
    pgu_org_mean = np.mean(pgus_org)
    pgi_org_std = np.std(pgis_org)
    pgu_org_std = np.std(pgus_org)
    pgi_mean_10 = np.mean(pgis_10)
    pgu_mean_10 = np.mean(pgus_10)
    pgi_std_10 = np.std(pgis_10)
    pgu_std_10 = np.std(pgus_10)
    pgi_org_mean_10 = np.mean(pgis_org_10)
    pgu_org_mean_10 = np.mean(pgus_org_10)
    pgi_org_std_10 = np.std(pgis_org_10)
    pgu_org_std_10 = np.std(pgus_org_10)
    fa_mean = np.mean(fas)
    fa_std = np.std(fas)
    if key == 'shapiq2':
        fa_interactions_mean = np.mean(fas_interactions)
        fa_interactions_std = np.std(fas_interactions)
    else:
        fa_interactions_mean = None
        fa_interactions_std = None
    metrics_dict[key] = {
        'pgi_mean': pgi_mean,
        'pgu_mean': pgu_mean,
        'pgi_std': pgi_std,
        'pgu_std': pgu_std,
        'pgi_mean_6': pgi_mean_10,
        'pgu_mean_6': pgu_mean_10,
        'pgi_std_6': pgi_std_10,
        'pgu_std_6': pgu_std_10,
        'pgi_org_mean': pgi_org_mean,
        'pgu_org_mean': pgu_org_mean,
        'pgi_org_std': pgi_org_std,
        'pgu_org_std': pgu_org_std,
        'pgi_org_mean_6': pgi_org_mean_10,
        'pgu_org_mean_6': pgu_org_mean_10,
        'pgi_org_std_6': pgi_org_std_10,
        'pgu_org_std_6': pgu_org_std_10,
        'fa_mean': fa_mean,
        'fa_std': fa_std,
        'fa_interactions_mean': fa_interactions_mean,
        'fa_interactions_std': fa_interactions_std
    }

    print(f"{key} PGI: {pgi_mean} ({pgi_std}), PGU: {pgu_mean} ({pgu_std})")
    print(f"PGI Org: {pgi_org_mean} ({pgi_org_std}), PGU Org: {pgu_org_mean} ({pgu_org_std})")
    print(f"PGI 10: {pgi_mean_10} ({pgi_std_10}), PGU 10: {pgu_mean_10} ({pgu_std_10})")
    print(f"PGI Org 10: {pgi_org_mean_10} ({pgi_org_std_10}), PGU Org 10: {pgu_org_mean_10} ({pgu_org_std_10})")
    print(f"Feature Agreement: {fa_mean} ({fa_std})")
    if key == 'shapiq2':
        print(f"Feature Agreement Interactions: {fa_interactions_mean} ({fa_interactions_std})")

    print('--' * 20)

lime


,features,abs_ranking
0,ecfp_feature_123,9.162088
1,ecfp_feature_30,5.483140
2,ecfp_feature_33,2.934667
3,ecfp_feature_16,1.276598
4,ecfp_feature_81,0.465590
5,ecfp_feature_10,0.259005
6,ecfp_feature_80,0.124999
7,ecfp_feature_90,0.115930
8,ecfp_feature_58,0.114165
9,ecfp_feature_99,0.110031


['ecfp_feature_123', 'ecfp_feature_30', 'ecfp_feature_33', 'ecfp_feature_16', 'ecfp_feature_81', 'ecfp_feature_10']
['ecfp_feature_123', 'ecfp_feature_30', 'ecfp_feature_33', 'ecfp_feature_16', 'ecfp_feature_81', 'ecfp_feature_10']
['ecfp_feature_123', 'ecfp_feature_30', 'ecfp_feature_33', 'ecfp_feature_16', 'ecfp_feature_81', 'ecfp_feature_10']
['ecfp_feature_123', 'ecfp_feature_30', 'ecfp_feature_33', 'ecfp_feature_16', 'ecfp_feature_81', 'ecfp_feature_10']
['ecfp_feature_123', 'ecfp_feature_30', 'ecfp_feature_33', 'ecfp_feature_16', 'ecfp_feature_81', 'ecfp_feature_10']
['ecfp_feature_123', 'ecfp_feature_30', 'ecfp_feature_33', 'ecfp_feature_16', 'ecfp_feature_81', 'ecfp_feature_10']
['ecfp_feature_123', 'ecfp_feature_30', 'ecfp_feature_33', 'ecfp_feature_16', 'ecfp_feature_81', 'ecfp_feature_10']
['ecfp_feature_123', 'ecfp_feature_30', 'ecfp_feature_33', 'ecfp_feature_16', 'ecfp_feature_81', 'ecfp_feature_10']
['ecfp_feature_123', 'ecfp_feature_30', 'ecfp_feature_33', 'ecfp_feature

,features,abs_ranking
0,ecfp_feature_123,7.411969
1,ecfp_feature_30,4.620494
2,ecfp_feature_10,2.754367
3,ecfp_feature_33,1.780950
4,ecfp_feature_16,1.071282
5,ecfp_feature_81,0.436088
6,ecfp_feature_22,0.006396
7,ecfp_feature_54,0.006305
8,ecfp_feature_121,0.006301
9,ecfp_feature_80,0.005994


['ecfp_feature_123', 'ecfp_feature_10', 'ecfp_feature_30', 'ecfp_feature_33', 'ecfp_feature_16', 'ecfp_feature_81']
['ecfp_feature_123', 'ecfp_feature_30', 'ecfp_feature_10', 'ecfp_feature_33', 'ecfp_feature_16', 'ecfp_feature_81']
['ecfp_feature_123', 'ecfp_feature_30', 'ecfp_feature_10', 'ecfp_feature_33', 'ecfp_feature_16', 'ecfp_feature_81']
['ecfp_feature_123', 'ecfp_feature_30', 'ecfp_feature_10', 'ecfp_feature_33', 'ecfp_feature_16', 'ecfp_feature_81']
['ecfp_feature_123', 'ecfp_feature_30', 'ecfp_feature_10', 'ecfp_feature_16', 'ecfp_feature_33', 'ecfp_feature_81']
['ecfp_feature_123', 'ecfp_feature_30', 'ecfp_feature_10', 'ecfp_feature_16', 'ecfp_feature_33', 'ecfp_feature_81']
['ecfp_feature_123', 'ecfp_feature_30', 'ecfp_feature_10', 'ecfp_feature_33', 'ecfp_feature_16', 'ecfp_feature_81']
['ecfp_feature_123', 'ecfp_feature_30', 'ecfp_feature_10', 'ecfp_feature_33', 'ecfp_feature_16', 'ecfp_feature_81']
['ecfp_feature_123', 'ecfp_feature_30', 'ecfp_feature_10', 'ecfp_feature

...

['ecfp_feature_123', 'ecfp_feature_10 x ecfp_feature_33', 'ecfp_feature_30', 'ecfp_feature_10 x ecfp_feature_123', 'ecfp_feature_10 x ecfp_feature_30', 'ecfp_feature_33']
['ecfp_feature_123', 'ecfp_feature_30', 'ecfp_feature_10 x ecfp_feature_123', 'ecfp_feature_78 x ecfp_feature_108', 'ecfp_feature_73 x ecfp_feature_105', 'ecfp_feature_12 x ecfp_feature_45']
['ecfp_feature_123', 'ecfp_feature_30', 'ecfp_feature_10 x ecfp_feature_123', 'ecfp_feature_10 x ecfp_feature_33', 'ecfp_feature_33', 'ecfp_feature_10 x ecfp_feature_30']
['ecfp_feature_123', 'ecfp_feature_30', 'ecfp_feature_10 x ecfp_feature_33', 'ecfp_feature_10 x ecfp_feature_123', 'ecfp_feature_10 x ecfp_feature_30', 'ecfp_feature_33']
['ecfp_feature_123', 'ecfp_feature_30', 'ecfp_feature_10 x ecfp_feature_123', 'ecfp_feature_33', 'ecfp_feature_78 x ecfp_feature_108', 'ecfp_feature_10 x ecfp_feature_33']
['ecfp_feature_123', 'ecfp_feature_30', 'ecfp_feature_10 x ecfp_feature_30', 'ecfp_feature_10 x ecfp_feature_123', 'ecfp_fea

,features,abs_ranking
0,ecfp_feature_123,0.601332
1,ecfp_feature_30,0.598835
2,ecfp_feature_33,0.537661
3,ecfp_feature_39,0.520599
4,ecfp_feature_10,0.519559
5,ecfp_feature_100,0.406783
6,ecfp_feature_80,0.381190
7,ecfp_feature_12,0.354141
8,ecfp_feature_86,0.346858
9,ecfp_feature_110,0.324802


meg counterfactual percent: (np.float64(0.9612), np.float64(0.36138272226104756))
['ecfp_feature_33', 'ecfp_feature_30', 'ecfp_feature_123', 'ecfp_feature_10', 'ecfp_feature_39', 'ecfp_feature_110']
['ecfp_feature_30', 'ecfp_feature_39', 'ecfp_feature_123', 'ecfp_feature_10', 'ecfp_feature_33', 'ecfp_feature_100']
['ecfp_feature_123', 'ecfp_feature_39', 'ecfp_feature_33', 'ecfp_feature_12', 'ecfp_feature_30', 'ecfp_feature_80']
['ecfp_feature_10', 'ecfp_feature_30', 'ecfp_feature_39', 'ecfp_feature_33', 'ecfp_feature_123', 'ecfp_feature_100']
['ecfp_feature_123', 'ecfp_feature_39', 'ecfp_feature_30', 'ecfp_feature_33', 'ecfp_feature_10', 'ecfp_feature_100']
['ecfp_feature_10', 'ecfp_feature_39', 'ecfp_feature_123', 'ecfp_feature_30', 'ecfp_feature_33', 'ecfp_feature_99']
['ecfp_feature_10', 'ecfp_feature_123', 'ecfp_feature_33', 'ecfp_feature_30', 'ecfp_feature_39', 'ecfp_feature_100']
['ecfp_feature_123', 'ecfp_feature_30', 'ecfp_feature_33', 'ecfp_feature_39', 'ecfp_feature_81', 'ecf

,features,abs_ranking
0,ecfp_feature_30,0.644297
1,ecfp_feature_123,0.600161
2,ecfp_feature_10,0.503628
3,ecfp_feature_39,0.478839
4,ecfp_feature_33,0.435711
5,ecfp_feature_80,0.365981
6,ecfp_feature_100,0.364369
7,ecfp_feature_16,0.360742
8,ecfp_feature_12,0.351673
9,ecfp_feature_45,0.343813


mmace counterfactual percent: (0.9924, np.float64(0.3200352223198329))
['ecfp_feature_30', 'ecfp_feature_123', 'ecfp_feature_10', 'ecfp_feature_33', 'ecfp_feature_39', 'ecfp_feature_110']
['ecfp_feature_30', 'ecfp_feature_39', 'ecfp_feature_123', 'ecfp_feature_16', 'ecfp_feature_10', 'ecfp_feature_100']
['ecfp_feature_123', 'ecfp_feature_30', 'ecfp_feature_39', 'ecfp_feature_80', 'ecfp_feature_12', 'ecfp_feature_10']
['ecfp_feature_10', 'ecfp_feature_30', 'ecfp_feature_39', 'ecfp_feature_123', 'ecfp_feature_3', 'ecfp_feature_16']
['ecfp_feature_123', 'ecfp_feature_30', 'ecfp_feature_39', 'ecfp_feature_10', 'ecfp_feature_100', 'ecfp_feature_80']
['ecfp_feature_123', 'ecfp_feature_30', 'ecfp_feature_39', 'ecfp_feature_99', 'ecfp_feature_10', 'ecfp_feature_33']
['ecfp_feature_30', 'ecfp_feature_123', 'ecfp_feature_10', 'ecfp_feature_100', 'ecfp_feature_1', 'ecfp_feature_58']
['ecfp_feature_30', 'ecfp_feature_123', 'ecfp_feature_33', 'ecfp_feature_39', 'ecfp_feature_80', 'ecfp_feature_81']

In [12]:
import pandas as pd
def convert_term_ranking_to_feature_ranking(ranking_with_interactions: list) -> list:
    """
    Converts a ranking of terms (features + interactions) into a ranking of
    only features based on their earliest appearance.
    """
    feature_to_best_rank = {}
    for i, term in enumerate(ranking_with_interactions):
        constituent_features = term.split(' x ')
        for feature in constituent_features:
            if feature not in feature_to_best_rank:
                feature_to_best_rank[feature] = i

    # Sort the features by their best rank
    sorted_features = sorted(feature_to_best_rank.items(), key=lambda item: item[1])

    return [feature for feature, rank in sorted_features]

def aggregate_rankings_by_mean_position(list_of_rankings: list) -> list:
    if not list_of_rankings:
        return []
    all_items = set()
    for ranking in list_of_rankings:
        all_items.update(ranking)
    item_scores = {}
    for item in all_items:
        positions = []
        for ranking in list_of_rankings:
            try:
                position = ranking.index(item)
            except ValueError:
                position = len(ranking)
            positions.append(position)
        item_scores[item] = np.mean(positions)
    sorted_items = sorted(item_scores.keys(), key=lambda item: item_scores[item])
    return sorted_items

In [13]:
pgis, pgus = [], []
pgis_org, pgus_org = [], []
pgis_org_10, pgus_org_10 = [], []
pgis_10, pgus_10 = [], []
fas = []
fas_interactions = []

for i in range(len(ranking_per_fold_dict['lime'])):

    model = os.path.join(model_dir, f'model_{i}.joblib')
    model = joblib.load(model)
    test_examples = results['test_data'][i].drop(columns=[target])
    train_examples = results['training_data'][i].drop(columns=[target])

    rankings = []
    for key in ranking_per_fold_dict.keys():
        ranking_current = list(ranking_per_fold_dict[key][i]['features'])
        if key == 'shapiq2':
            ranking_current = convert_term_ranking_to_feature_ranking(ranking_current)
        rankings.append(ranking_current)

    aggregated_ranking = aggregate_rankings_by_mean_position(rankings)
    pgi_one, pgi_org = pgi(test_examples, aggregated_ranking, model, train_examples)
    pgu_one, pgu_org = pgu(test_examples, aggregated_ranking, model, train_examples)
    pgi_ten, pgi_ten_org = pgi(test_examples, aggregated_ranking, model, train_examples, len_max=6)
    pgu_ten, pgu_ten_org = pgu(test_examples, aggregated_ranking, model, train_examples, len_max=6)
    pgis.append(pgi_one)
    pgus.append(pgu_one)
    pgis_10.append(pgi_ten)
    pgus_10.append(pgu_ten)
    pgis_org.append(pgi_org)
    pgus_org.append(pgu_org)
    pgis_org_10.append(pgi_ten_org)
    pgus_org_10.append(pgu_ten_org)

    fa = feature_agreement(choices['qm9_simple_linear6'], aggregated_ranking)
    fas.append(fa)

pgi_mean = np.mean(pgis)
pgu_mean = np.mean(pgus)
pgi_std = np.std(pgis)
pgu_std = np.std(pgus)
pgi_org_mean = np.mean(pgis_org)
pgu_org_mean = np.mean(pgus_org)
pgi_org_std = np.std(pgis_org)
pgu_org_std = np.std(pgus_org)
pgi_mean_10 = np.mean(pgis_10)
pgu_mean_10 = np.mean(pgus_10)
pgi_std_10 = np.std(pgis_10)
pgu_std_10 = np.std(pgus_10)
pgi_org_mean_10 = np.mean(pgis_org_10)
pgu_org_mean_10 = np.mean(pgus_org_10)
pgi_org_std_10 = np.std(pgis_org_10)
pgu_org_std_10 = np.std(pgus_org_10)
fa_mean = np.mean(fas)
fa_std = np.std(fas)
fa_interactions_mean = None
fa_interactions_std = None


metrics_dict['aggregated'] = {
        'pgi_mean': pgi_mean,
        'pgu_mean': pgu_mean,
        'pgi_std': pgi_std,
        'pgu_std': pgu_std,
        'pgi_mean_6': pgi_mean_10,
        'pgu_mean_6': pgu_mean_10,
        'pgi_std_6': pgi_std_10,
        'pgu_std_6': pgu_std_10,
        'pgi_org_mean': pgi_org_mean,
        'pgu_org_mean': pgu_org_mean,
        'pgi_org_std': pgi_org_std,
        'pgu_org_std': pgu_org_std,
        'pgi_org_mean_6': pgi_org_mean_10,
        'pgu_org_mean_6': pgu_org_mean_10,
        'pgi_org_std_6': pgi_org_std_10,
        'pgu_org_std_6': pgu_org_std_10,
        'fa_mean': fa_mean,
        'fa_std': fa_std,
        'fa_interactions_mean': fa_interactions_mean,
        'fa_interactions_std': fa_interactions_std
    }

print(f"Aggregated PGI: {pgi_mean} ({pgi_std}), PGU: {pgu_mean} ({pgu_std})")
print(f"Aggregated PGI Org: {pgi_org_mean} ({pgi_org_std}), PGU Org: {pgu_org_mean} ({pgu_org_std})")
print(f"Aggregated PGI 10: {pgi_mean_10} ({pgi_std_10}), PGU 10: {pgu_mean_10} ({pgu_std_10})")
print(f"Aggregated PGI Org 10: {pgi_org_mean_10} ({pgi_org_std_10}), PGU Org 10: {pgu_org_mean_10} ({pgu_org_std_10})")

Aggregated PGI: 0.37730304332113795 (0.06877851760964068), PGU: 0.021574986886307126 (0.004723935278966684)
Aggregated PGI Org: 13.640488304317367 (1.4608718335645312), PGU Org: 0.7835937492786893 (0.13040568893954035)
Aggregated PGI 10: 0.3736767371715746 (0.0725470737536571), PGU 10: 0.001534745650611311 (0.0026061345189241423)
Aggregated PGI Org 10: 13.508575691740656 (1.7280141000269305), PGU Org 10: 0.05434019979564255 (0.08809853361860223)


In [14]:
# Save the results
os.makedirs(os.path.join(results_dir, 'analysis'), exist_ok=True)
with open(os.path.join(results_dir, 'analysis', 'metrics_results.pickle'), 'wb') as f:
    pickle.dump(metrics_dict, f)
with open(os.path.join(results_dir, 'analysis', 'metrics_top10_results.pickle'), 'wb') as f:
    pickle.dump(metrics_top10_dict, f)
with open(os.path.join(results_dir, 'analysis', 'ranking_results.pickle'), 'wb') as f:
    pickle.dump(ranking_dict, f)
with open(os.path.join(results_dir, 'analysis', 'ranking_per_fold_results.pickle'), 'wb') as f:
    pickle.dump(ranking_per_fold_dict, f)
with open(os.path.join(results_dir, 'analysis', 'cf_validity_results.pickle'), 'wb') as f:
    pickle.dump(cf_validity_dict, f)
with open(os.path.join(results_dir, 'analysis', 'cf_similarity_results.pickle'), 'wb') as f:
    pickle.dump(cf_similarity_dict, f)

In [15]:
from src.analysis.xai_eval import rank_correlation

def convert_term_ranking_to_feature_ranking(ranking_with_interactions: list) -> list:
    """
    Converts a ranking of terms (features + interactions) into a ranking of
    only features based on their earliest appearance.
    """
    feature_to_best_rank = {}
    for i, term in enumerate(ranking_with_interactions):
        constituent_features = term.split(' x ')
        for feature in constituent_features:
            if feature not in feature_to_best_rank:
                feature_to_best_rank[feature] = i

    # Sort the features by their best rank
    sorted_features = sorted(feature_to_best_rank.items(), key=lambda item: item[1])

    return [feature for feature, rank in sorted_features]

#calculate rankings correlations
pairs_of_ranks = [(key1, key2) for key1 in ranking_per_fold_dict.keys() for key2 in ranking_per_fold_dict.keys()]
correlations = {}
for key1, key2 in pairs_of_ranks:
    corrs = []
    for i in range(len(ranking_per_fold_dict[key1])):
        rank1 = ranking_per_fold_dict[key1][i]['features']
        rank2 = ranking_per_fold_dict[key2][i]['features']
        if key1 == 'shapiq2':
            rank1 = convert_term_ranking_to_feature_ranking(rank1)
        if key2 == 'shapiq2':
            rank2 = convert_term_ranking_to_feature_ranking(rank2)
        c = rank_correlation(rank1, rank2).statistic
        corrs.append(c)
    correlation = np.mean(corrs)
    correlations[(key1, key2)] = correlation
    print(f"{key1} vs {key2}: {correlation:.4f}")
    print('--' * 20)

with open(os.path.join(results_dir, 'analysis', 'correlations_results.pickle'), 'wb') as f:
    pickle.dump(correlations, f)

lime vs lime: 1.0000
----------------------------------------
lime vs shap: 0.4380
----------------------------------------
lime vs shapiq1: 0.3961
----------------------------------------
lime vs shapiq2: 0.1796
----------------------------------------
lime vs meg: 0.2269
----------------------------------------
lime vs mmace: 0.2936
----------------------------------------
shap vs lime: 0.4380
----------------------------------------
shap vs shap: 1.0000
----------------------------------------
shap vs shapiq1: 0.4653
----------------------------------------
shap vs shapiq2: 0.3202
----------------------------------------
shap vs meg: 0.2618
----------------------------------------
shap vs mmace: 0.3172
----------------------------------------
shapiq1 vs lime: 0.3961
----------------------------------------
shapiq1 vs shap: 0.4653
----------------------------------------
shapiq1 vs shapiq1: 1.0000
----------------------------------------
shapiq1 vs shapiq2: 0.5083
-------------------

In [16]:
#calculate rankings correlations
pairs_of_ranks = [(key1, key2) for key1 in ranking_per_fold_dict.keys() for key2 in ranking_per_fold_dict.keys()]
correlations_top10 = {}
for key1, key2 in pairs_of_ranks:
    corrs = []
    for i in range(len(ranking_per_fold_dict[key1])):
        rank1 = ranking_per_fold_dict[key1][i]['features']
        rank2 = ranking_per_fold_dict[key2][i]['features']
        if key1 == 'shapiq2':
            rank1 = convert_term_ranking_to_feature_ranking(rank1)
        if key2 == 'shapiq2':
            rank2 = convert_term_ranking_to_feature_ranking(rank2)
        c = rank_correlation(rank1, rank2, k=6).statistic
        corrs.append(c)
    correlation = np.mean(corrs)
    correlations_top10[(key1, key2)] = correlation
    print(f"{key1} vs {key2}: {correlation:.4f}")
    print('--' * 20)

with open(os.path.join(results_dir, 'analysis', 'correlations_top6_results.pickle'), 'wb') as f:
    pickle.dump(correlations_top10, f)

lime vs lime: 1.0000
----------------------------------------
lime vs shap: 0.6229
----------------------------------------
lime vs shapiq1: 0.6229
----------------------------------------
lime vs shapiq2: 0.5897
----------------------------------------
lime vs meg: 0.3160
----------------------------------------
lime vs mmace: 0.2908
----------------------------------------
shap vs lime: 0.6229
----------------------------------------
shap vs shap: 1.0000
----------------------------------------
shap vs shapiq1: 1.0000
----------------------------------------
shap vs shapiq2: 0.8494
----------------------------------------
shap vs meg: 0.4817
----------------------------------------
shap vs mmace: 0.4807
----------------------------------------
shapiq1 vs lime: 0.6229
----------------------------------------
shapiq1 vs shap: 1.0000
----------------------------------------
shapiq1 vs shapiq1: 1.0000
----------------------------------------
shapiq1 vs shapiq2: 0.8494
-------------------